# Dynamic Embedded Topic Model (DETM) for UN General Debates

## Overview
This notebook implements the Dynamic Embedded Topic Model from Dieng et al. (2019) "Topic Modeling in Embedding Spaces" applied to the UN General Debates corpus.

### Paper Reference
Dieng, A. B., Ruiz, F. J., & Blei, D. M. (2019). Topic modeling in embedding spaces. Transactions of the Association for Computational Linguistics, 7, 439-453.

### Model Architecture
**Generative Process:**
- Topic embeddings evolve over time: α_t ~ N(α_{t-1}, σ²I)
- Topic-word distributions: β_k,t = softmax(α_k,t · ρ) where ρ are word embeddings
- Document topic proportions: θ_t ~ N(0, I) (Gaussian prior)
- Word generation: w ~ Categorical(softmax(β^T θ))

**Variational Inference:**
- Approximate posterior: q(θ_t | w_t) = N(μ_t, Σ_t)
- Encoder network outputs μ_t and log σ_t from bag-of-words representation
- ELBO = E_q[log p(w|θ,α,ρ)] - KL(q(θ)||p(θ))

### Implementation Structure
1. **Data Processing**: Load, clean, and prepare UN debates corpus
   - UN-specific stopwords + WordNet lemmatization
   - IDF-ranked vocabulary selection (replaces raw-frequency ranking)
   - Tighter frequency gates (MIN_DF=30, MAX_DF=0.3)
2. **Embeddings**: Word2Vec skip-gram trained on corpus (matches Dieng et al.)
   - Replaces SentenceTransformer; W2V geometry is designed for inner-product factorisation
3. **Model Components**: Encoder, Decoder, Temporal Dynamics
4. **Loss Modules**: KL Divergence and Reconstruction Loss (modular)
5. **Training**: Optimization loop with ELBO maximization
6. **Evaluation**: Coherence, perplexity, topic quality metrics
7. **Visualization**: TensorBoard real-time monitoring (metrics, loss curves, gradients)


---
## 1. Setup and Imports

In [1]:
# Standard library imports
import os
import sys
import json
import pickle
import warnings
from pathlib import Path
from collections import Counter, defaultdict
from typing import Dict, List, Tuple, Optional, Union
from contextlib import nullcontext

# Load environment variables (Kaggle API credentials)
from dotenv import load_dotenv
load_dotenv()  # This loads variables from .env file

# Verify Kaggle credentials are loaded
if os.getenv('KAGGLE_USERNAME') and os.getenv('KAGGLE_KEY'):
    print("✓ Kaggle credentials loaded successfully")
else:
    print("⚠ Kaggle credentials not found. Make sure .env file contains KAGGLE_USERNAME and KAGGLE_KEY")

# Data processing
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.special import softmax

# Deep learning
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam, AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.tensorboard import SummaryWriter

# NLP libraries
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
nltk.download('punkt_tab', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
from transformers import AutoTokenizer, AutoModel
import gensim
from gensim.models import Word2Vec
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora import Dictionary

# Utilities
from tqdm.auto import tqdm
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.manifold import TSNE
import umap

# Set random seeds for reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("✓ All imports loaded successfully")


✓ Kaggle credentials loaded successfully
✓ All imports loaded successfully


---
## 2. Configuration and Hyperparameters

In [2]:
class Config:
    """Central configuration for DETM experiment"""
    
    # Paths
    DATA_DIR = Path('../data')
    MODELS_DIR = Path('../models')
    OUTPUTS_DIR = Path('../outputs')
    
    # Data preprocessing
    MIN_DF = 30       # Minimum document frequency (raised from 10 → 30; ~0.4% of 7507 docs)
    MAX_DF = 0.3      # Maximum document frequency (lowered from 0.5 → 0.3; removes UN boilerplate)
    MAX_VOCAB_SIZE = 10000
    MIN_DOC_LENGTH = 15  # Minimum vocabulary words per document (applied after cleaning)
    
    # Model architecture
    NUM_TOPICS = 50  # K in the paper
    # Word2Vec skip-gram trained on corpus (replaces SentenceTransformer)
    # Skip-gram inner-product geometry directly matches DETM's β = softmax(α·ρ^T)
    EMBEDDING_DIM = 300  # Word2Vec standard dimension (was 384 for SentenceTransformer)
    W2V_WINDOW = 5       # Word2Vec context window
    W2V_EPOCHS = 10      # Word2Vec training epochs
    # NOTE: Word embeddings (ρ) are W2V skip-gram, raw (no L2-norm), TRAINABLE in DETMDecoder
    
    # Temporal Baseline Encoder (LSTM-based structured inference for η_t)
    COMPRESSION_DIM = 400  # Compression of BoW before LSTM
    LSTM_LAYERS = 4  # Number of LSTM layers
    LSTM_HIDDEN = 400  # Hidden units in each LSTM layer
    
    # Document Topic Encoder (Amortized inference for θ_d)
    DOC_HIDDEN_DIM = 800  # Hidden dimension for document encoder MLP
    DOC_DROPOUT = 0.0  # Dropout for document encoder (0.0 matches original — dropout in VAE encoders causes collapse)
    
    # Topic Embeddings (Mean-field inference for α^(t))
    # No neural network - just trainable parameters (T × K × L)
    INIT_ALPHA_STD = 1.0  # Initialization std for α parameters (matches original torch.randn default)
    # Initialize logvar to 0 (i.e. variance=1, matching the randn init scale in the original)
    INIT_ALPHA_LOGVAR = 0.0
    
    # Legacy parameters (kept for compatibility)
    HIDDEN_DIM = 512  # Deprecated - use DOC_HIDDEN_DIM
    NUM_ENCODER_LAYERS = 2  # Deprecated
    DROPOUT = 0.2  # Deprecated - use DOC_DROPOUT
    
    # Temporal dynamics - Gaussian random walk prior variances (from Dieng et al. 2019)
    USE_TEMPORAL = True
    TEMPORAL_TYPE = 'gaussian'  # Deprecated
    ETA_PRIOR_VARIANCE   = 0.005  # δ²: prior variance for η random walk  p(η_t | η_{t-1}) = N(η_{t-1}, δ²I)
    ALPHA_PRIOR_VARIANCE = 0.005  # γ²: prior variance for α random walk  p(α_t | α_{t-1}) = N(α_{t-1}, γ²I)
    TEMPORAL_VARIANCE = 0.005  # Deprecated alias - use ETA_PRIOR_VARIANCE or ALPHA_PRIOR_VARIANCE
    
    # Training — aligned with Dieng et al. 2019
    BATCH_SIZE = 1000  # Original uses 1000; small batches destabilize global η LSTM
    NUM_EPOCHS = 400        # Paper trains longer; early stopping (patience=15) terminates when appropriate
    LEARNING_RATE = 0.005   # Original default with batch_size=1000
    WEIGHT_DECAY = 1.2e-6   # From Dieng et al. 2019 (used with Adam, not AdamW)
    WARMUP_EPOCHS = 10      # Deprecated — no scheduler
    CLIP_GRAD = 2.0         # From Dieng et al. 2019
    
    # Loss weights
    RECON_WEIGHT = 1.0  # Weight for reconstruction term
    KL_THETA_WEIGHT = 1.0  # Weight for KL(q(θ) || p(θ)) - document topics
    KL_ETA_WEIGHT = 1.0  # Weight for KL(q(η) || p(η)) - temporal baselines (normalized by D)
    KL_ALPHA_WEIGHT = 1.0  # Weight for KL(q(α) || p(α)) - topic embeddings (normalized by D)
    KL_WEIGHT = 1.0  # Deprecated - legacy total KL weight
    
    # Evaluation
    TOP_N_WORDS = [10, 15, 20]  # Top-N words for coherence
    COHERENCE_METRICS = ['c_v', 'c_npmi']  # Coherence types
    
    # Checkpointing
    SAVE_EVERY = 10  # Save checkpoint every N epochs
    PATIENCE = 15  # Early stopping patience
    
    def __init__(self):
        # Create directories if they don't exist
        self.DATA_DIR.mkdir(exist_ok=True)
        self.MODELS_DIR.mkdir(exist_ok=True)
        self.OUTPUTS_DIR.mkdir(exist_ok=True)

config = Config()
print("Configuration initialized successfully")
print(f"  MIN_DF={config.MIN_DF}, MAX_DF={config.MAX_DF}, EMBEDDING_DIM={config.EMBEDDING_DIM}")


Configuration initialized successfully
  MIN_DF=30, MAX_DF=0.3, EMBEDDING_DIM=300


In [3]:
# Device configuration (for DETM model training - not for embeddings)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device for DETM training: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"Available GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print()

Using device for DETM training: cuda
GPU: NVIDIA GeForce RTX 4070 Laptop GPU
CUDA Version: 12.8
Available GPU memory: 8.59 GB



---
## 3. Data Loading and Preprocessing

### 3.1 Download and Load UN General Debates Dataset

In [4]:
# check if data is already downloaded
data_file = config.DATA_DIR / 'un-general-debates.csv'
if data_file.exists():
    print(f"✓ Data file already exists at {data_file}")
else:
    !kaggle datasets download -d 'unitednations/un-general-debates' -p ../data --unzip

✓ Data file already exists at ../data/un-general-debates.csv


In [5]:
def load_and_explore_data():
    """Load and perform initial exploration of UN debates data"""
    
    # Load dataset
    csv_path = config.DATA_DIR / 'un-general-debates.csv'
    df = pd.read_csv(csv_path)
    
    print("=" * 60)
    print("UN GENERAL DEBATES DATASET")
    print("=" * 60)
    print(f"\nDataset shape: {df.shape}")
    print(f"\nColumns: {df.columns.tolist()}")
    print(f"\nData types:\n{df.dtypes}")
    print(f"\nMissing values:\n{df.isnull().sum()}")
    
    # Date range
    if 'year' in df.columns:
        print(f"\nYear range: {df['year'].min()} - {df['year'].max()}")
        print(f"Years in dataset: {sorted(df['year'].unique())}")
    
    # Countries
    if 'country' in df.columns:
        print(f"\nNumber of countries: {df['country'].nunique()}")
        print(f"\nTop 10 countries by number of speeches:")
        print(df['country'].value_counts().head(10))
    
    # Text statistics
    if 'text' in df.columns:
        df['text_length'] = df['text'].astype(str).str.split().str.len()
        print(f"\nText length statistics:")
        print(df['text_length'].describe())
        
        # Note: Visualizations are handled by TensorBoard during training
        # For data exploration, refer to the printed statistics above
    
    return df

# Load and explore
df_raw = load_and_explore_data()

UN GENERAL DEBATES DATASET

Dataset shape: (7507, 4)

Columns: ['session', 'year', 'country', 'text']

Data types:
session     int64
year        int64
country    object
text       object
dtype: object

Missing values:
session    0
year       0
country    0
text       0
dtype: int64

Year range: 1970 - 2015
Years in dataset: [np.int64(1970), np.int64(1971), np.int64(1972), np.int64(1973), np.int64(1974), np.int64(1975), np.int64(1976), np.int64(1977), np.int64(1978), np.int64(1979), np.int64(1980), np.int64(1981), np.int64(1982), np.int64(1983), np.int64(1984), np.int64(1985), np.int64(1986), np.int64(1987), np.int64(1988), np.int64(1989), np.int64(1990), np.int64(1991), np.int64(1992), np.int64(1993), np.int64(1994), np.int64(1995), np.int64(1996), np.int64(1997), np.int64(1998), np.int64(1999), np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), n

### 3.2 Data Preprocessor Class

In [6]:
class DataPreprocessor:
    """
    Preprocesses UN General Debates corpus for DETM.
    
    Operations:
    - Text cleaning: lowercase, NLTK tokenization, stopword removal (NLTK + UN-specific),
      alpha-only filter, lemmatization (WordNetLemmatizer), length filter
    - Vocabulary building: document-frequency gates (MIN_DF / MAX_DF), IDF-ranked selection
    - Bag-of-words representation
    - Temporal ordering
    
    Key changes vs. original:
    - UN-specific boilerplate stopwords added (diplomacy formulaic phrases)
    - WordNetLemmatizer: collapses morphological variants (develop/developing/development → develop)
    - IDF-ranked vocab selection (replaces raw-frequency ranking):
        retains discriminative words rather than the most common ones
    - MIN_DF raised (10→30), MAX_DF lowered (0.5→0.3) for tighter frequency gates
    - Length filter applied on cleaned vocab tokens (not raw whitespace split)
    """

    # UN General Debates domain-specific stopwords.
    # These are formulaic diplomatic phrases that appear across nearly all speeches
    # and carry zero topic discrimination signal.
    UN_STOPWORDS = {
        # Organisational / procedural
        "united", "nations", "assembly", "general", "security", "council",
        "president", "secretary", "minister", "delegation", "representative",
        "distinguished", "session", "resolution", "agenda", "committee",
        "conference", "plenary", "headquarter", "secretariat",
        # Diplomatic boilerplate
        "reaffirm", "reiterate", "congratulate", "behalf", "honour", "welcome",
        "express", "wish", "convey", "utilize", "utilize", "hereby",
        "take", "made", "shall", "would", "could", "must", "may",
        # Overly generic
        "international", "world", "countries", "states", "government",
        "republic", "peoples", "nation", "global", "member",
        "also", "well", "one", "two", "new", "first", "many",
        "year", "years", "within", "since", "among", "like",
        # Common filler that NLTK misses
        "said", "make", "need", "call", "called", "issue", "issues",
        "important", "ensure", "support", "continue", "including",
        "area", "areas", "level", "levels", "part", "role",
        "assembly", "council", "president", "delegation", "session", "minister", "republic"
    }

    def __init__(self, config):
        self.config = config
        # Combine NLTK English stopwords with UN-specific ones
        self.stopwords = set(stopwords.words('english')) | self.UN_STOPWORDS
        self.lemmatizer = WordNetLemmatizer()
        self.vocabulary = None
        self.word2idx = None
        self.idx2word = None

    def clean_text(self, text: str) -> List[str]:
        """
        Clean and tokenize a single document.
        
        Pipeline: lowercase → word_tokenize → alpha-only filter → stopword removal
                  → length filter (>2) → lemmatization
        """
        text = str(text).lower()
        tokens = word_tokenize(text)
        tokens = [
            self.lemmatizer.lemmatize(token)
            for token in tokens
            if token.isalpha()
            and len(token) > 2
            and token not in self.stopwords
        ]
        return tokens

    def build_vocabulary(self, documents: List[List[str]]) -> Dict:
        """
        Build vocabulary with frequency filtering and IDF-ranked selection.
        
        Frequency gates:
            MIN_DF: word must appear in at least MIN_DF documents (removes rare noise)
            MAX_DF: word must appear in at most MAX_DF fraction of documents (removes omnipresent boilerplate)
        
        Selection criterion: IDF score (descending) — keeps the most discriminative words.
            IDF(w) = log((N+1) / (df(w)+1))
        Previously the selection was by raw corpus frequency (ascending commonness),
        which preferentially retained the least informative words.
        """
        print("Building vocabulary...")

        word_freq = Counter()
        doc_freq = Counter()

        for doc in tqdm(documents, desc="Counting frequencies"):
            word_freq.update(doc)
            doc_freq.update(set(doc))

        num_docs = len(documents)
        min_df_count = self.config.MIN_DF
        max_df_count = int(self.config.MAX_DF * num_docs)

        valid_words = [
            word for word, df in doc_freq.items()
            if min_df_count <= df <= max_df_count
        ]

        # Sort by IDF (descending): high IDF = discriminative, low IDF = ubiquitous
        idf_scores = {
            w: np.log((num_docs + 1) / (doc_freq[w] + 1))
            for w in valid_words
        }
        sorted_words = sorted(
            valid_words,
            key=lambda w: idf_scores[w],
            reverse=True
        )[:self.config.MAX_VOCAB_SIZE]

        self.word2idx = {word: idx for idx, word in enumerate(sorted_words)}
        self.idx2word = {idx: word for word, idx in self.word2idx.items()}
        self.vocabulary = sorted_words

        print(f"\nVocabulary size: {len(self.vocabulary)}")
        print(f"Total unique tokens before filtering: {len(word_freq)}")
        print(f"Tokens passing df gates [{min_df_count}, {max_df_count}]: {len(valid_words)}")
        print(f"Final vocabulary (top-{self.config.MAX_VOCAB_SIZE} by IDF): {len(self.vocabulary)}")

        return {
            'word2idx': self.word2idx,
            'idx2word': self.idx2word,
            'vocabulary': self.vocabulary,
            'word_freq': {w: word_freq[w] for w in self.vocabulary},
            'doc_freq': {w: doc_freq[w] for w in self.vocabulary},
            'idf_scores': {w: idf_scores[w] for w in self.vocabulary},
        }

    def doc_to_bow(self, tokens: List[str]) -> np.ndarray:
        """Convert tokenized document to bag-of-words vector."""
        bow = np.zeros(len(self.vocabulary), dtype=np.float32)
        for token in tokens:
            if token in self.word2idx:
                bow[self.word2idx[token]] += 1
        return bow

    def _create_temporal_index(self, df: pd.DataFrame, bow_matrix: np.ndarray) -> Dict:
        """
        Create temporal indexing structure for DETM.
        
        Maps documents to time steps and computes average BoW per time period
        for the LSTM temporal baseline encoder.
        """
        if 'year' not in df.columns:
            print("Warning: No 'year' column found. Creating single time step.")
            return {
                'time_steps': [0],
                'doc_to_time': np.zeros(len(df), dtype=np.int64),
                'time_to_docs': {0: list(range(len(df)))},
                'avg_bow_per_time': (bow_matrix.sum(axis=0, keepdims=True) / len(bow_matrix)),
                'num_time_steps': 1,
            }

        time_steps = sorted(df['year'].unique())
        time_to_idx = {year: idx for idx, year in enumerate(time_steps)}
        doc_to_time = np.array([time_to_idx[year] for year in df['year']], dtype=np.int64)

        time_to_docs = {t_idx: [] for t_idx in range(len(time_steps))}
        for doc_idx, t_idx in enumerate(doc_to_time):
            time_to_docs[t_idx].append(doc_idx)

        avg_bow_per_time = np.zeros((len(time_steps), bow_matrix.shape[1]), dtype=np.float32)
        for t_idx, doc_indices in time_to_docs.items():
            if len(doc_indices) > 0:
                time_docs = bow_matrix[doc_indices]
                normalized_docs = time_docs / (time_docs.sum(axis=1, keepdims=True) + 1e-10)
                avg_bow_per_time[t_idx] = normalized_docs.mean(axis=0)

        print(f"\n{'='*60}")
        print("TEMPORAL STRUCTURE")
        print(f"{'='*60}")
        print(f"Number of time steps (T): {len(time_steps)}")
        print(f"Time range: {time_steps[0]} - {time_steps[-1]}")
        print(f"Average documents per time step: {len(df) / len(time_steps):.1f}")
        print(f"Min docs in time step: {min(len(docs) for docs in time_to_docs.values())}")
        print(f"Max docs in time step: {max(len(docs) for docs in time_to_docs.values())}")

        return {
            'time_steps': time_steps,
            'doc_to_time': doc_to_time,
            'time_to_docs': time_to_docs,
            'avg_bow_per_time': avg_bow_per_time,
            'num_time_steps': len(time_steps),
        }

    def preprocess_corpus(self, df: pd.DataFrame) -> Dict:
        """
        Complete preprocessing pipeline.
        
        Returns:
            Dictionary with:
            - bow_matrix: Document-term matrix (N × V, float32)
            - tokens_list: List of tokenized documents (cleaned, lemmatized)
            - vocabulary_info: vocab / word2idx / idf_scores etc.
            - metadata: Filtered DataFrame
            - temporal_info: Temporal structure for DETM LSTM
        """
        print("\n" + "=" * 60)
        print("PREPROCESSING CORPUS")
        print("=" * 60)
        print(f"Stopwords: {len(self.stopwords)} total "
              f"({len(set(stopwords.words('english')))} NLTK + "
              f"{len(self.UN_STOPWORDS)} UN-specific)")

        # Step 1: Tokenize + clean all documents (no length filter yet)
        print("\nTokenizing and cleaning documents...")
        tokens_list_raw = []
        for text in tqdm(df['text'], desc="Tokenizing"):
            tokens_list_raw.append(self.clean_text(text))

        # Step 2: Apply length filter on cleaned tokens (not raw whitespace split)
        min_len = self.config.MIN_DOC_LENGTH
        valid_mask = [len(t) >= min_len for t in tokens_list_raw]
        df_filtered = df[valid_mask].reset_index(drop=True)
        tokens_list = [t for t, keep in zip(tokens_list_raw, valid_mask) if keep]
        print(f"Documents after cleaned-token length filter (≥{min_len}): {len(df_filtered)} "
              f"(removed {sum(1 for v in valid_mask if not v)})")

        # Step 3: Build vocabulary
        vocab_info = self.build_vocabulary(tokens_list)

        # Step 4: Convert to bag-of-words; drop docs with no vocab words
        print("\nConverting to bag-of-words representation...")
        bow_matrix = []
        valid_indices = []
        for idx, tokens in enumerate(tqdm(tokens_list, desc="Creating BoW")):
            bow = self.doc_to_bow(tokens)
            if bow.sum() > 0:
                bow_matrix.append(bow)
                valid_indices.append(idx)

        bow_matrix = np.array(bow_matrix, dtype=np.float32)
        df_filtered = df_filtered.iloc[valid_indices].reset_index(drop=True)
        tokens_list = [tokens_list[i] for i in valid_indices]

        print(f"\nFinal number of documents: {len(bow_matrix)}")
        print(f"Vocabulary size: {bow_matrix.shape[1]}")
        print(f"Average vocab words per document: {bow_matrix.sum(axis=1).mean():.2f}")
        print(f"Sparsity: {(bow_matrix == 0).sum() / bow_matrix.size * 100:.2f}%")

        # Step 5: Sort by temporal order
        if 'year' in df_filtered.columns:
            sort_idx = df_filtered['year'].argsort()
            bow_matrix = bow_matrix[sort_idx]
            df_filtered = df_filtered.iloc[sort_idx].reset_index(drop=True)
            tokens_list = [tokens_list[i] for i in sort_idx]
            print(f"Documents sorted by year: {df_filtered['year'].min()} → {df_filtered['year'].max()}")

        # Step 6: Build temporal index
        temporal_info = self._create_temporal_index(df_filtered, bow_matrix)

        processed_data = {
            'bow_matrix': bow_matrix,
            'tokens_list': tokens_list,
            'vocabulary_info': vocab_info,
            'metadata': df_filtered,
            'num_docs': len(bow_matrix),
            'vocab_size': len(self.vocabulary),
            'temporal_info': temporal_info,
        }

        save_path = self.config.DATA_DIR / 'preprocessed_data.pkl'
        with open(save_path, 'wb') as f:
            pickle.dump(processed_data, f)
        print(f"\nPreprocessed data saved to: {save_path}")

        return processed_data


print("DataPreprocessor class defined.")


DataPreprocessor class defined.


In [7]:
# Usage example:
preprocessor = DataPreprocessor(config)
processed_data = preprocessor.preprocess_corpus(df_raw)


PREPROCESSING CORPUS
Stopwords: 277 total (198 NLTK + 79 UN-specific)

Tokenizing and cleaning documents...


Tokenizing:   0%|          | 0/7507 [00:00<?, ?it/s]

Documents after cleaned-token length filter (≥15): 7507 (removed 0)
Building vocabulary...


Counting frequencies:   0%|          | 0/7507 [00:00<?, ?it/s]


Vocabulary size: 10000
Total unique tokens before filtering: 42959
Tokens passing df gates [30, 2252]: 10108
Final vocabulary (top-10000 by IDF): 10000

Converting to bag-of-words representation...


Creating BoW:   0%|          | 0/7507 [00:00<?, ?it/s]


Final number of documents: 7507
Vocabulary size: 10000
Average vocab words per document: 496.74
Sparsity: 96.19%
Documents sorted by year: 1970 → 2015

TEMPORAL STRUCTURE
Number of time steps (T): 46
Time range: 1970 - 2015
Average documents per time step: 163.2
Min docs in time step: 70
Max docs in time step: 195

Preprocessed data saved to: ../data/preprocessed_data.pkl


### 3.3 Vocabulary Quality Diagnostic

Verify the vocabulary looks correct before training:
- Top words (highest IDF) should be specific domain terms, **not** UN boilerplate
- Known boilerplate words ("united", "nations", "assembly"…) should be absent
- Lemmatization should have collapsed morphological variants

In [8]:
# ── Vocabulary Quality Diagnostic ────────────────────────────────────────────
vocab = preprocessor.vocabulary
vocab_info = processed_data['vocabulary_info']
word_freq = vocab_info['word_freq']
doc_freq = vocab_info['doc_freq']
idf_scores = vocab_info['idf_scores']
n_docs = processed_data['num_docs']

print('=' * 70)
print('VOCABULARY QUALITY CHECK')
print('=' * 70)
print(f'Vocabulary size: {len(vocab):,}')
print(f'Total documents: {n_docs:,}')

print('\n--- Top 50 words by IDF (most discriminative) ---')
for i, w in enumerate(vocab[:50]):
    print(f'  {i+1:>3}. {w:<22} idf={idf_scores[w]:.3f}  '
          f'df={doc_freq[w]:>5} ({doc_freq[w]/n_docs*100:.1f}%)')

print('\n--- Bottom 20 words by IDF (most common — gates should keep these useful) ---')
for i, w in enumerate(vocab[-20:]):
    rank = len(vocab) - 20 + i + 1
    print(f'  {rank:>5}. {w:<22} idf={idf_scores[w]:.3f}  '
          f'df={doc_freq[w]:>5} ({doc_freq[w]/n_docs*100:.1f}%)')

print('\n--- Boilerplate words that should NOT be in the vocabulary ---')
boilerplate_check = [
    'united', 'nations', 'assembly', 'general', 'security', 'council',
    'president', 'delegation', 'distinguished', 'session', 'resolution',
    'international', 'minister', 'republic',
]
found  = [w for w in boilerplate_check if w in preprocessor.word2idx]
absent = [w for w in boilerplate_check if w not in preprocessor.word2idx]
print(f'  ✓ Correctly removed ({len(absent)}): {absent}')
if found:
    print(f'  ⚠ Still present ({len(found)}) — consider adding to UN_STOPWORDS: {found}')
else:
    print('  ✓ All boilerplate words successfully filtered!')

print('\n--- Lemmatisation check (variants should collapse to one stem) ---')
lemma_groups = [
    ('develop', 'development', 'developing', 'developed', 'developmental'),
    ('terror', 'terrorism', 'terrorist'),
    ('economic', 'economy', 'economies'),
    ('nuclear',),
    ('peace', 'peaceful', 'peacefully'),
]
for group in lemma_groups:
    present = [w for w in group if w in preprocessor.word2idx]
    print(f'  {group[0]:>15}: surviving forms in vocab → {present}')

print('\n--- 20 random vocabulary words (sanity check) ---')
rng = np.random.default_rng(42)
idx_sample = rng.choice(len(vocab), size=20, replace=False)
print(' ', ', '.join([vocab[int(i)] for i in sorted(idx_sample)]))

VOCABULARY QUALITY CHECK
Vocabulary size: 10,000
Total documents: 7,507

--- Top 50 words by IDF (most discriminative) ---
    1. inhibiting             idf=5.490  df=   30 (0.4%)
    2. cap                    idf=5.490  df=   30 (0.4%)
    3. boosting               idf=5.490  df=   30 (0.4%)
    4. fetter                 idf=5.490  df=   30 (0.4%)
    5. sharpened              idf=5.490  df=   30 (0.4%)
    6. heterogeneous          idf=5.490  df=   30 (0.4%)
    7. prom                   idf=5.490  df=   30 (0.4%)
    8. firmest                idf=5.490  df=   30 (0.4%)
    9. amenity                idf=5.490  df=   30 (0.4%)
   10. freight                idf=5.490  df=   30 (0.4%)
   11. ominously              idf=5.490  df=   30 (0.4%)
   12. egregious              idf=5.490  df=   30 (0.4%)
   13. rupture                idf=5.490  df=   30 (0.4%)
   14. impatiently            idf=5.490  df=   30 (0.4%)
   15. surmounting            idf=5.490  df=   30 (0.4%)
   16. unfailingly    

---
## 4. Word Embeddings Generation

### 4.1 Word2Vec Embedding Generator

**Why Word2Vec instead of SentenceTransformer?**

DETM computes topic-word distributions as `β_k = softmax(α_k · ρ^T)`.
For topics to be distinct, word embeddings ρ must be **angularly spread** across the embedding space so that different α_k vectors point towards different clusters.

- **SentenceTransformer (all-MiniLM-L6-v2)** encodes single words using 6 transformer layers + mean-pool. Single-word outputs cluster in a narrow angular cone (avg pairwise cosine ~0.65–0.80 for unrelated words) because the representation is dominated by positional/syntactic context frames, not distributional co-occurrence geometry. This makes all β_k nearly identical regardless of α_k → topic collapse.

- **Word2Vec skip-gram** directly optimises the inner-product objective: *given word w, predict context words*. The resulting geometry is specifically designed for inner-product factorisation — exactly what DETM's `α · ρ^T` assumes. Embeddings are much more uniformly distributed over the sphere (avg pairwise cosine ~0.05–0.20).

Training on the preprocessing corpus ensures the embedding space reflects the actual vocabulary distribution of UN speeches.


In [9]:
class EmbeddingGenerator:
    """
    Generates word embeddings using Word2Vec skip-gram trained on the corpus.
    
    Replaces the previous SentenceTransformer approach. Rationale:
    
    DETM's decoder computes β_k = softmax(α_k · ρ^T).  After L2-normalising ρ,
    this becomes β_k,w ∝ exp(||α_k|| · cos(α_k, ρ_w)).  For topics to be distinct,
    word embeddings must be spread across the angular space — a property that
    Word2Vec skip-gram provides directly (its training objective optimises exactly
    the inner-product factorisation that DETM needs).
    
    SentenceTransformer single-word embeddings cluster in a narrow cone
    (avg pairwise cosine ~0.65–0.80) because they capture syntactic/positional
    context, not co-occurrence geometry.  This causes posterior collapse.
    
    Architecture (matching Dieng et al. 2019):
        - Algorithm: skip-gram  (sg=1)
        - Vector size: 300  (config.EMBEDDING_DIM)
        - Window: 5         (config.W2V_WINDOW)
        - min_count: same as config.MIN_DF (consistent with vocab gates)
        - Epochs: 10        (config.W2V_EPOCHS)
    
    Caching: trained Word2Vec model is saved to DATA_DIR/word2vec.model and
    reloaded on subsequent runs, avoiding unnecessary retraining.
    Similarly, the extracted embedding matrix is saved to DATA_DIR/word_embeddings.npy.
    
    OOV handling: words in the vocabulary that are not in the W2V model (extremely
    rare since both use the same cleaned corpus) receive a small random unit vector.
    """

    W2V_FILENAME = 'word2vec.model'
    EMB_FILENAME  = 'word_embeddings.npy'

    def __init__(self, config):
        self.config = config
        self.model = None
        self.embedding_dim = config.EMBEDDING_DIM

    def _w2v_path(self) -> Path:
        return self.config.DATA_DIR / self.W2V_FILENAME

    def _emb_path(self) -> Path:
        return self.config.DATA_DIR / self.EMB_FILENAME

    def train_word2vec(self, tokens_list: List[List[str]]) -> 'Word2Vec':
        """
        Train Word2Vec skip-gram on the tokenized corpus and save to disk.
        
        Args:
            tokens_list: List of tokenized documents (already cleaned + lemmatized)
        
        Returns:
            Trained gensim Word2Vec model
        """
        print("\n" + "=" * 60)
        print("TRAINING WORD2VEC SKIP-GRAM")
        print("=" * 60)
        print(f"Corpus size: {len(tokens_list):,} documents")
        print(f"Total tokens: {sum(len(t) for t in tokens_list):,}")
        print(f"Parameters: vector_size={self.embedding_dim}, "
              f"window={self.config.W2V_WINDOW}, "
              f"min_count={self.config.MIN_DF}, "
              f"sg=1 (skip-gram), "
              f"epochs={self.config.W2V_EPOCHS}")

        np.random.seed(SEED)
        self.model = Word2Vec(
            sentences=tokens_list,
            vector_size=self.embedding_dim,
            window=self.config.W2V_WINDOW,
            min_count=self.config.MIN_DF,  # consistent with vocab frequency gate
            workers=4,
            sg=1,    # skip-gram (vs. CBOW)
            seed=SEED,
            epochs=self.config.W2V_EPOCHS,
        )
        save_path = self._w2v_path()
        self.model.save(str(save_path))
        print(f"✓ Word2Vec trained — vocabulary: {len(self.model.wv):,} words")
        print(f"✓ Model saved to: {save_path}")
        return self.model

    def load_word2vec(self) -> 'Word2Vec':
        """Load a previously saved Word2Vec model from disk."""
        path = self._w2v_path()
        self.model = Word2Vec.load(str(path))
        print(f"✓ Word2Vec loaded from cache: {path} "
              f"({len(self.model.wv):,} words)")
        return self.model

    def generate_vocabulary_embeddings(
        self,
        vocabulary: List[str],
        tokens_list: Optional[List[List[str]]] = None,
        force_retrain: bool = False,
    ) -> np.ndarray:
        """
        Extract vocabulary embeddings, using cached files when available.
        
        Cache behaviour:
          - If DATA_DIR/word2vec.model exists AND force_retrain=False → load cached W2V.
          - Otherwise train from scratch (tokens_list required in that case).
          - Extracted embedding matrix is always saved to DATA_DIR/word_embeddings.npy.
        
        Args:
            vocabulary:    Ordered vocabulary list (from DataPreprocessor)
            tokens_list:   Tokenized corpus — required only when training from scratch
            force_retrain: If True, ignore any cached model and retrain
        
        Returns:
            Embedding matrix (vocab_size, embedding_dim), float32
        """
        w2v_path = self._w2v_path()

        if not force_retrain and w2v_path.exists():
            print(f"✓ Cached Word2Vec model found — loading (skip training).")
            print(f"  To retrain, call with force_retrain=True or delete {w2v_path}")
            self.load_word2vec()
        else:
            if tokens_list is None:
                raise ValueError(
                    "tokens_list is required to train Word2Vec. "
                    f"No cached model found at {w2v_path}."
                )
            self.train_word2vec(tokens_list)

        print("\n" + "=" * 60)
        print("EXTRACTING VOCABULARY EMBEDDINGS")
        print("=" * 60)
        print(f"Vocabulary size: {len(vocabulary):,}")

        embeddings = np.zeros((len(vocabulary), self.embedding_dim), dtype=np.float32)
        oov_words = []

        for idx, word in enumerate(vocabulary):
            if word in self.model.wv:
                embeddings[idx] = self.model.wv[word]
            else:
                # OOV: random unit vector (extremely rare — same min_count applied)
                oov_vec = np.random.randn(self.embedding_dim).astype(np.float32)
                embeddings[idx] = oov_vec / (np.linalg.norm(oov_vec) + 1e-10)
                oov_words.append(word)

        if oov_words:
            print(f"⚠ OOV words ({len(oov_words)}): {oov_words[:10]}"
                  f"{'...' if len(oov_words) > 10 else ''}")
        else:
            print("✓ All vocabulary words found in Word2Vec model (0 OOV)")

        print(f"\nEmbedding matrix shape: {embeddings.shape}")
        print(f"Embedding statistics (before L2-norm):")
        print(f"  Mean: {embeddings.mean():.4f}")
        print(f"  Std:  {embeddings.std():.4f}")
        print(f"  Min:  {embeddings.min():.4f}")
        print(f"  Max:  {embeddings.max():.4f}")

        # Save raw embeddings (DETMDecoder will L2-normalise them at init)
        emb_path = self._emb_path()
        np.save(emb_path, embeddings)
        print(f"✓ Embeddings saved to: {emb_path}")

        return embeddings


print("EmbeddingGenerator class defined (Word2Vec skip-gram, with caching).")


EmbeddingGenerator class defined (Word2Vec skip-gram, with caching).


In [10]:
# Train (or load cached) Word2Vec skip-gram and extract vocabulary embeddings.
# Pass tokens_list so training can run on first execution;
# subsequent runs load the saved model from data/word2vec.model automatically.
embedding_gen = EmbeddingGenerator(config)
word_embeddings = embedding_gen.generate_vocabulary_embeddings(
    preprocessor.vocabulary,
    tokens_list=processed_data['tokens_list'],  # ignored if cached model exists
)


✓ Cached Word2Vec model found — loading (skip training).
  To retrain, call with force_retrain=True or delete ../data/word2vec.model
✓ Word2Vec loaded from cache: ../data/word2vec.model (11,084 words)

EXTRACTING VOCABULARY EMBEDDINGS
Vocabulary size: 10,000
✓ All vocabulary words found in Word2Vec model (0 OOV)

Embedding matrix shape: (10000, 300)
Embedding statistics (before L2-norm):
  Mean: -0.0061
  Std:  0.2275
  Min:  -1.4018
  Max:  1.3503
✓ Embeddings saved to: ../data/word_embeddings.npy


### 4.2 Embedding Quality Diagnostic

Key checks before training:
- **Angular spread**: average pairwise cosine similarity should be low (~0.05–0.20 for Word2Vec vs ~0.65–0.80 for SentenceTransformer). Low spread = topics can differentiate.
- **Nearest neighbours**: semantically related words should cluster together.
- **Post-L2-norm geometry**: since `DETMDecoder` L2-normalises ρ, we check the normalised dot-product distribution.

In [11]:
# ── Embedding Quality Diagnostic ─────────────────────────────────────────────
rng = np.random.default_rng(42)
n_sample = 500  # number of random pairs for cosine similarity estimate

# L2-normalise once (mirrors what DETMDecoder does at init)
norms = np.linalg.norm(word_embeddings, axis=1, keepdims=True) + 1e-10
emb_norm = word_embeddings / norms

print('=' * 70)
print('EMBEDDING QUALITY DIAGNOSTIC  (Word2Vec skip-gram)')
print('=' * 70)

# 1. Angular spread: sample random pairs and compute cosine similarity
idx_a = rng.integers(0, len(preprocessor.vocabulary), size=n_sample)
idx_b = rng.integers(0, len(preprocessor.vocabulary), size=n_sample)
# avoid self-pairs
mask = idx_a != idx_b
idx_a, idx_b = idx_a[mask], idx_b[mask]
cos_sims = (emb_norm[idx_a] * emb_norm[idx_b]).sum(axis=1)

print(f'\nPairwise cosine similarity ({len(cos_sims)} random pairs):')
print(f'  Mean:   {cos_sims.mean():.4f}   (Word2Vec target: 0.05–0.20)')
print(f'  Median: {np.median(cos_sims):.4f}')
print(f'  Std:    {cos_sims.std():.4f}')
print(f'  p10:    {np.percentile(cos_sims, 10):.4f}')
print(f'  p90:    {np.percentile(cos_sims, 90):.4f}')
if cos_sims.mean() < 0.3:
    print('  ✓ Embeddings are well-spread (low pairwise similarity)')
else:
    print('  ⚠ Embeddings may be clustered — expected <0.3 for Word2Vec')

# 2. Nearest-neighbour sanity checks
probe_words = ['nuclear', 'trade', 'climate', 'terror', 'develop', 'conflict',
               'poverty', 'democraci', 'disarmament', 'environment']
print('\nNearest neighbours (top-5, by cosine similarity):')
w2v = embedding_gen.model
for word in probe_words:
    # try lemmatized form too
    from nltk.stem import WordNetLemmatizer
    lemma = WordNetLemmatizer().lemmatize(word)
    query = lemma if lemma in w2v.wv else word
    if query in w2v.wv:
        neighbours = w2v.wv.most_similar(query, topn=5)
        nn_str = ', '.join([f"{w}({s:.2f})" for w, s in neighbours])
        print(f'  {query:<16} → {nn_str}')
    else:
        print(f'  {query:<16} → not in W2V vocabulary')

# 3. Embedding norm distribution
raw_norms = np.linalg.norm(word_embeddings, axis=1)
print(f'\nRaw embedding L2-norm distribution:')
print(f'  Mean: {raw_norms.mean():.4f}  Std: {raw_norms.std():.4f}  '
      f'Min: {raw_norms.min():.4f}  Max: {raw_norms.max():.4f}')
print(f'  (After L2-norm in DETMDecoder all norms → 1.0)')

EMBEDDING QUALITY DIAGNOSTIC  (Word2Vec skip-gram)

Pairwise cosine similarity (500 random pairs):
  Mean:   0.0916   (Word2Vec target: 0.05–0.20)
  Median: 0.0859
  Std:    0.0716
  p10:    0.0043
  p90:    0.1776
  ✓ Embeddings are well-spread (low pairwise similarity)

Nearest neighbours (top-5, by cosine similarity):
  nuclear          → weapon(0.61), atomic(0.59), npt(0.58), testing(0.56), nonproliferation(0.54)
  trade            → trading(0.59), gatt(0.56), wto(0.54), tariff(0.53), liberalization(0.51)
  climate          → change(0.61), warming(0.57), kyoto(0.49), environment(0.49), unfccc(0.49)
  terror           → terrorist(0.51), violence(0.51), terrorism(0.48), repression(0.48), vile(0.45)
  develop          → strengthen(0.53), build(0.49), establish(0.46), expand(0.46), promote(0.46)
  conflict         → dispute(0.53), flare(0.51), flashpoint(0.50), strife(0.49), erupting(0.48)
  poverty          → hunger(0.73), illiteracy(0.66), underdevelopment(0.62), disease(0.59), malnu

---
## 5. DETM Model Architecture

### 5.1 Document Topic Encoder (Amortized Inference with Logistic-Normal)

In [12]:
class DocumentTopicEncoder(nn.Module):
    """
    Amortized Variational Inference for Document Topic Proportions (θ_d).
    
    Uses MLP-based amortized inference conditioned on both document text and
    temporal baseline. Outputs logistic-normal distribution (softmax of Gaussian).
    
    Architecture:
        Inputs: 
            - w_d: Document BoW (V)
            - η_{t_d}: Temporal baseline for document's year (K)
        ↓
        Concatenate: [w_d, η_{t_d}] → (V + K)
        ↓
        MLP: 
            - Linear(V+K → 800) → ReLU → Dropout(0.1)
            - Linear(800 → 800) → ReLU → Dropout(0.1)
        ↓
        Two parallel outputs:
            - μ_θ: Linear(800 → K)
            - log σ²_θ: Linear(800 → K)
        ↓
        Reparameterization:
            - γ_d = μ_θ + ε ⊙ exp(0.5 · log σ²_θ)
            - θ_d = softmax(γ_d)  # logistic-normal transformation
    """
    
    def __init__(
        self,
        vocab_size: int,
        num_topics: int,
        hidden_dim: int = 800,
        dropout: float = 0.1
    ):
        super().__init__()
        
        self.vocab_size = vocab_size
        self.num_topics = num_topics
        self.hidden_dim = hidden_dim
        
        # Input is concatenated [BoW (V), eta (K)]
        input_dim = vocab_size + num_topics
        
        # MLP with 2 hidden layers (no dropout — matches original enc_drop=0.0)
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )
        
        # Output layers for mean and log variance
        self.fc_mu = nn.Linear(hidden_dim, num_topics)
        self.fc_logvar = nn.Linear(hidden_dim, num_topics)
    
    def reparameterize(self, mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
        """
        Reparameterization trick: γ = μ + σ * ε where ε ~ N(0, I)
        
        Args:
            mu: Mean (batch_size, num_topics)
            logvar: Log variance (batch_size, num_topics)
        
        Returns:
            Sampled γ (batch_size, num_topics) - NOT YET ON SIMPLEX
        """
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    
    def forward(
        self,
        bow: torch.Tensor,
        eta: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Forward pass through document topic encoder.
        
        Args:
            bow: Bag-of-words tensor (batch_size, vocab_size)
            eta: Temporal baseline (batch_size, num_topics)
        
        Returns:
            theta: Topic proportions on simplex (batch_size, num_topics)
            mu: Mean of pre-softmax Gaussian (batch_size, num_topics)
            logvar: Log variance of pre-softmax Gaussian (batch_size, num_topics)
        """
        # Normalize BoW (simple L1-norm, matching original — no BatchNorm)
        bow_norm = bow / (bow.sum(dim=1, keepdim=True) + 1e-10)
        
        # Concatenate BoW and temporal baseline
        combined = torch.cat([bow_norm, eta], dim=1)  # (batch_size, V + K)
        
        # Encode
        hidden = self.encoder(combined)  # (batch_size, 800)
        
        # Compute variational parameters
        mu = self.fc_mu(hidden)  # (batch_size, K)
        logvar = self.fc_logvar(hidden)  # (batch_size, K)
        
        # Sample from Gaussian
        gamma = self.reparameterize(mu, logvar)  # (batch_size, K)
        
        # Apply logistic-normal transformation: θ = softmax(γ)
        theta = F.softmax(gamma, dim=-1)  # (batch_size, K) - now on simplex
        
        return theta, mu, logvar


### 5.2 Decoder (Generative Model with External Topic Embeddings)

In [13]:
class DETMDecoder(nn.Module):
    """
    Generative network for DETM.
    
    Models word generation given topic proportions and topic embeddings:
        p(w | θ, α, ρ) where:
        - θ: document topic proportions (batch_size, num_topics)
        - α: topic embeddings (num_topics, embedding_dim) or (batch_size, num_topics, embedding_dim)
        - ρ: word embeddings (vocab_size, embedding_dim)  ← raw W2V, trainable
    
    Topic-word distribution: β_k = softmax(α_k · ρ^T)
    The raw W2V inner-product geometry is preserved (no L2-normalization).
    Word embeddings are trainable by default, matching the original implementation.
    
    Word probability: p(w) = θ^T · β
    
    Note: Topic embeddings α are NOT stored here - they are passed as input.
          This allows for time-varying α^(t) from mean-field parameters.
    """
    
    def __init__(
        self,
        num_topics: int,
        vocab_size: int,
        embedding_dim: int,
        word_embeddings: torch.Tensor,
        train_word_embeddings: bool = False
    ):
        super().__init__()
        
        self.num_topics = num_topics
        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        
        # Store raw word embeddings (NO L2-normalization — matches original).
        # L2-norm compresses logit dynamic range and with small α init causes
        # β ≈ uniform → posterior collapse.  Raw W2V norms encode word importance.
        self.word_embeddings = nn.Parameter(
            word_embeddings.clone(),
            requires_grad=train_word_embeddings
        )
    
    def get_beta(self, topic_embeddings: torch.Tensor) -> torch.Tensor:
        """
        Compute topic-word distribution matrix β.
        
        β[k, w] = p(w | topic=k) = softmax_w(α_k · ρ_w)
        
        Args:
            topic_embeddings: Topic embeddings α, shape:
                - (num_topics, embedding_dim) for shared α across batch
                - (batch_size, num_topics, embedding_dim) for time-varying α
        
        Returns:
            beta: Topic-word distribution of shape (num_topics, vocab_size)
                  or (batch_size, num_topics, vocab_size)
        """
        if topic_embeddings.dim() == 2:
            # Shared topic embeddings: (K, L)
            # Compute logits: α · ρ^T = (K, L) × (L, V) = (K, V)
            logits = torch.mm(topic_embeddings, self.word_embeddings.t())
            # Apply softmax over vocabulary dimension
            beta = F.softmax(logits, dim=-1)  # (K, V)
            
        elif topic_embeddings.dim() == 3:
            # Time-varying topic embeddings: (B, K, L)
            # Compute logits: α · ρ^T = (B, K, L) × (L, V) = (B, K, V)
            logits = torch.matmul(topic_embeddings, self.word_embeddings.t())
            # Apply softmax over vocabulary dimension
            beta = F.softmax(logits, dim=-1)  # (B, K, V)
        else:
            raise ValueError(f"Invalid topic_embeddings shape: {topic_embeddings.shape}")
        
        return beta
    
    def forward(
        self,
        theta: torch.Tensor,
        topic_embeddings: torch.Tensor
    ) -> torch.Tensor:
        """
        Generate word distribution for documents.
        
        Args:
            theta: Document topic proportions (batch_size, num_topics)
                   Already on simplex (from logistic-normal)
            topic_embeddings: Topic embeddings α, shape:
                - (num_topics, embedding_dim) for static/shared
                - (batch_size, num_topics, embedding_dim) for time-varying
        
        Returns:
            word_dist: Word probability distribution (batch_size, vocab_size)
        """
        # Get topic-word distribution
        beta = self.get_beta(topic_embeddings)  # (K, V) or (B, K, V)
        
        # Compute document-word distribution
        # p(w | d) = θ^T · β
        
        if beta.dim() == 2:
            # Shared β: (K, V)
            # θ: (B, K), β: (K, V) → (B, K) × (K, V) = (B, V)
            word_dist = torch.mm(theta, beta)
        else:
            # Time-varying β: (B, K, V)
            # θ: (B, K, 1), β: (B, K, V) → (B, K, 1) × (B, K, V) = (B, V)
            word_dist = torch.bmm(theta.unsqueeze(1), beta).squeeze(1)
        
        return word_dist

print("DETMDecoder class defined.")


DETMDecoder class defined.


### 5.3 Temporal Baseline Encoder (LSTM-based Structured Inference)

In [14]:
class TemporalBaselineEncoder(nn.Module):
    """
    LSTM-based Structured Variational Inference for Global Temporal Baselines (η_t).
    
    This component models the overall topic popularity across the corpus for each
    time step using a structured variational family.
    
    Architecture:
        Input: w̃_t (V-dimensional average normalized BoW for year t)
        ↓
        Compression: Linear(V → 400)
        ↓
        LSTM: 4 layers, 400 hidden units each
        ↓
        For each time step t:
            - Residual: Concat [LSTM_output_t (400), η_{t-1} (K)]
            - Two parallel Linear layers → μ_η (K), log σ²_η (K)
            - Sample: η_t = μ_η + ε ⊙ exp(0.5 · log σ²_η)
            - Compute KL(q(η_t) || p(η_t | η_{t-1})) using the FRESH η_{t-1} sample
    
    KL is computed *during* the forward pass so that η_{t-1} is still part of the
    live computation graph — this enables Monte Carlo gradients without graph reuse.
    
    Prior: p(η_t | η_{t-1}) = N(η_{t-1}, δ²I)  where δ² = ETA_PRIOR_VARIANCE
    """
    
    def __init__(
        self,
        vocab_size: int,
        num_topics: int,
        compression_dim: int = 400,
        lstm_layers: int = 4,
        lstm_hidden: int = 400,
        delta_sq: float = 0.005  # δ²: prior variance for the random walk
    ):
        super().__init__()
        
        self.vocab_size = vocab_size
        self.num_topics = num_topics
        self.compression_dim = compression_dim
        self.lstm_layers = lstm_layers
        self.lstm_hidden = lstm_hidden
        self.delta_sq = delta_sq  # δ²: p(η_t | η_{t-1}) = N(η_{t-1}, δ²I)
        
        # Compression layer: V → 400
        self.compress = nn.Linear(vocab_size, compression_dim)
        
        # LSTM: sequential over time steps
        self.lstm = nn.LSTM(
            input_size=compression_dim,
            hidden_size=lstm_hidden,
            num_layers=lstm_layers,
            batch_first=True,
            dropout=0.0  # Original uses eta_dropout=0.0; dropout on η-LSTM destabilizes temporal prior
        )
        
        # Output layers: Input is [LSTM_output (400), η_{t-1} (K)] → 400 + K
        self.fc_mu = nn.Linear(lstm_hidden + num_topics, num_topics)
        self.fc_logvar = nn.Linear(lstm_hidden + num_topics, num_topics)
        
        # eta_timeline cached for inference use (get_document_topics)
        self.eta_timeline = None
    
    def reparameterize(self, mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    
    def forward(
        self,
        avg_bow_timeline: torch.Tensor,
        return_params: bool = False
    ) -> torch.Tensor:
        """
        Process entire timeline to generate global temporal baselines.
        
        Computes KL divergence *during* the loop so that η_{t-1} is a fresh
        sample in the current computation graph (Monte Carlo, matches paper).
        
        Prior: p(η_0) = N(0, δ²I),  p(η_t | η_{t-1}) = N(η_{t-1}, δ²I)
        
        KL(N(μ, σ²) || N(m, δ²)) = 0.5 * [σ²/δ² + (μ-m)²/δ² - 1 - log(σ²/δ²)]
        """
        T = avg_bow_timeline.shape[0]
        
        # Compress: (T, V) → (T, compression_dim)
        compressed = self.compress(avg_bow_timeline)
        
        # LSTM over full timeline
        lstm_input = compressed.unsqueeze(0)   # (1, T, compression_dim)
        lstm_output, _ = self.lstm(lstm_input) # (1, T, lstm_hidden)
        lstm_output = lstm_output.squeeze(0)   # (T, lstm_hidden)
        
        eta_mu_list = []
        eta_logvar_list = []
        eta_list = []
        kl_terms = []
        
        # η_prev starts at zero (used as the prior mean for t=0)
        eta_prev = torch.zeros(1, self.num_topics, device=avg_bow_timeline.device)
        
        for t in range(T):
            lstm_t = lstm_output[t:t+1]                         # (1, lstm_hidden)
            combined = torch.cat([lstm_t, eta_prev], dim=1)     # (1, lstm_hidden + K)
            
            mu_t     = self.fc_mu(combined)                     # (1, K)
            logvar_t = self.fc_logvar(combined)                 # (1, K)
            eta_t    = self.reparameterize(mu_t, logvar_t)      # (1, K)
            
            # KL(q(η_t) || p(η_t | η_prev)) = KL(N(μ_t, σ²_t) || N(η_prev, δ²I))
            # For t=0: η_prev = 0, so this is KL(N(μ_0, σ²_0) || N(0, δ²I))
            var_t = torch.exp(logvar_t)
            kl_t = 0.5 * torch.sum(
                var_t / self.delta_sq
                + (mu_t - eta_prev).pow(2) / self.delta_sq
                - 1.0
                - torch.log(var_t / self.delta_sq)
            )
            kl_terms.append(kl_t)
            
            # eta_prev for the next step is the FRESH sample (still in graph)
            eta_prev = eta_t
            
            eta_mu_list.append(mu_t)
            eta_logvar_list.append(logvar_t)
            eta_list.append(eta_t)
        
        eta_mu_timeline    = torch.cat(eta_mu_list,    dim=0)  # (T, K)
        eta_logvar_timeline = torch.cat(eta_logvar_list, dim=0) # (T, K)
        eta_timeline       = torch.cat(eta_list,       dim=0)  # (T, K)
        
        # Cache eta_timeline for inference-time use (e.g. get_document_topics)
        self.eta_timeline = eta_timeline
        
        # kl_eta is a live tensor — gradients flow back to LSTM/fc_mu/fc_logvar
        kl_eta = sum(kl_terms)
        
        if return_params:
            return eta_timeline, eta_mu_timeline, eta_logvar_timeline, kl_eta
        return eta_timeline, kl_eta

print("TemporalBaselineEncoder class defined.")


TemporalBaselineEncoder class defined.


---
## 6. Loss Modules (Modular Design)

### 6.1 Reconstruction Loss

In [15]:
class ReconstructionLoss(nn.Module):
    """
    Computes reconstruction loss for DETM.
    
    Measures how well the model reconstructs the observed word counts.
    
    Loss = -E_q(θ)[log p(w | θ, α, ρ)]
    
    Two common choices:
    1. Multinomial: Standard for topic models
    2. Poisson: Alternative that can handle over-dispersion
    
    We use multinomial with log-likelihood:
        log p(w) = sum_v n_v * log p(v)
    where n_v is count of word v, p(v) is predicted probability
    """
    
    def __init__(self, loss_type: str = 'multinomial'):
        super().__init__()
        self.loss_type = loss_type
    
    def multinomial_loss(
        self,
        bow: torch.Tensor,
        word_dist: torch.Tensor
    ) -> torch.Tensor:
        """
        Multinomial negative log-likelihood.
        
        Args:
            bow: Observed word counts (batch_size, vocab_size)
            word_dist: Predicted word distribution (batch_size, vocab_size)
        
        Returns:
            Negative log-likelihood per sample (batch_size,)
        """
        # Add small epsilon to avoid log(0)
        word_dist = torch.clamp(word_dist, min=1e-10, max=1.0)
        
        # Negative log-likelihood: -sum_v n_v * log p_v
        nll = -(bow * torch.log(word_dist)).sum(dim=-1)
        
        return nll
    
    def poisson_loss(
        self,
        bow: torch.Tensor,
        word_dist: torch.Tensor
    ) -> torch.Tensor:
        """
        Poisson negative log-likelihood.
        
        Args:
            bow: Observed word counts (batch_size, vocab_size)
            word_dist: Predicted word rates (batch_size, vocab_size)
        
        Returns:
            Negative log-likelihood per sample (batch_size,)
        """
        # Get document lengths (total word count per document)
        doc_lengths = bow.sum(dim=-1, keepdim=True)
        
        # Scale word distribution by document length to get rates
        rates = word_dist * doc_lengths
        rates = torch.clamp(rates, min=1e-10)
        
        # Poisson NLL: sum_v (λ_v - n_v * log λ_v + log n_v!)
        # Ignore factorial term (constant wrt parameters)
        nll = (rates - bow * torch.log(rates)).sum(dim=-1)
        
        return nll
    
    def forward(
        self,
        bow: torch.Tensor,
        word_dist: torch.Tensor,
        return_breakdown: bool = False
    ) -> Union[torch.Tensor, Dict[str, torch.Tensor]]:
        """
        Compute reconstruction loss.
        
        Args:
            bow: Observed word counts (batch_size, vocab_size)
            word_dist: Predicted word distribution (batch_size, vocab_size)
            return_breakdown: If True, return dict with per-sample loss
        
        Returns:
            Reconstruction loss (scalar) or dict with breakdown
        """
        if self.loss_type == 'multinomial':
            recon_loss = self.multinomial_loss(bow, word_dist)
        elif self.loss_type == 'poisson':
            recon_loss = self.poisson_loss(bow, word_dist)
        else:
            raise ValueError(f"Unknown loss type: {self.loss_type}")
        
        # Average over batch
        recon_loss_mean = recon_loss.mean()
        
        if return_breakdown:
            return {
                'recon_total': recon_loss_mean,
                'recon_per_sample': recon_loss
            }
        else:
            return recon_loss_mean

print("ReconstructionLoss class defined.")

ReconstructionLoss class defined.


---
## 7. Complete DETM Model

### 7.1 Full DETM Implementation

In [16]:
class DETM(nn.Module):
    """
    Complete Dynamic Embedded Topic Model with Full Architecture.
    
    Combines three distinct variational families:
    1. TemporalBaselineEncoder: LSTM structured inference for η_t
    2. DocumentTopicEncoder:    MLP amortized inference for θ_d (logistic-normal)
    3. Mean-field parameters:   Trainable α^(t)  (T × K × L)
    
    ELBO: E_q[log p(w|θ,α,ρ)] - KL(q(θ)||p(θ)) - KL(q(η)||p(η)) - KL(q(α)||p(α))
    
    Priors (Gaussian random walks from Dieng et al. 2019):
        p(η_0) = N(0, δ²I),  p(η_t | η_{t-1}) = N(η_{t-1}, δ²I)   δ² = ETA_PRIOR_VARIANCE
        p(α_0) = N(0, γ²I),  p(α_t | α_{t-1}) = N(α_{t-1}, γ²I)   γ² = ALPHA_PRIOR_VARIANCE
        p(θ_d) = Logistic-Normal(η_{t_d}, I)   (η is the PRIOR MEAN, not just conditioning)
    
    Loss normalization:
        - recon_loss and kl_theta are per-document means (averaged over batch)
        - kl_eta and kl_alpha are global sums (over T×K and T×K×L respectively)
          → divided by num_train_docs so all terms are on a per-document scale
    
    The LSTM is re-run on every forward call (matching Dieng et al. reference impl).
    This is cheap (T~50 steps) vs encoding a batch of documents, and keeps the full
    gradient graph intact — no graph-reuse RuntimeErrors.
    """
    
    def __init__(
        self,
        config,
        word_embeddings: torch.Tensor,
        num_time_steps: int,
        avg_bow_timeline: np.ndarray,
        num_train_docs: int = 1
    ):
        super().__init__()
        
        self.config = config
        self.num_time_steps = num_time_steps
        # Used to normalize global KL terms (kl_eta, kl_alpha) to per-document scale,
        # matching the per-document averaging of recon_loss and kl_theta.
        self.num_train_docs = num_train_docs
        
        vocab_size, embedding_dim = word_embeddings.shape
        
        # Store average BoW timeline for LSTM (as buffer, not parameter)
        self.register_buffer('avg_bow_timeline', torch.FloatTensor(avg_bow_timeline))
        
        # Component 1: Temporal Baseline Encoder
        self.temporal_baseline_encoder = TemporalBaselineEncoder(
            vocab_size=vocab_size,
            num_topics=config.NUM_TOPICS,
            compression_dim=config.COMPRESSION_DIM,
            lstm_layers=config.LSTM_LAYERS,
            lstm_hidden=config.LSTM_HIDDEN,
            delta_sq=config.ETA_PRIOR_VARIANCE  # δ²
        )
        
        # Component 2: Document Topic Encoder
        self.doc_encoder = DocumentTopicEncoder(
            vocab_size=vocab_size,
            num_topics=config.NUM_TOPICS,
            hidden_dim=config.DOC_HIDDEN_DIM,
            dropout=config.DOC_DROPOUT
        )
        
        # Component 3: Topic Embeddings (mean-field, no neural network)
        # Shape: (T, K, L) — time steps × topics × embedding dimension
        self.alpha_mu = nn.Parameter(
            torch.randn(num_time_steps, config.NUM_TOPICS, embedding_dim) * config.INIT_ALPHA_STD
        )
        self.alpha_logvar = nn.Parameter(
            torch.ones(num_time_steps, config.NUM_TOPICS, embedding_dim) * config.INIT_ALPHA_LOGVAR
        )
        
        # Component 4: Decoder (word embeddings trainable — matches original train_embeddings=1 default)
        self.decoder = DETMDecoder(
            num_topics=config.NUM_TOPICS,
            vocab_size=vocab_size,
            embedding_dim=embedding_dim,
            word_embeddings=word_embeddings,
            train_word_embeddings=True
        )
        
        self.recon_loss_fn = ReconstructionLoss(loss_type='multinomial')
    
    def sample_alpha(self, time_indices: torch.Tensor) -> torch.Tensor:
        """
        Sample topic embeddings α^(t) via reparameterization.

        At eval time, returns the posterior mean (no noise) for stable inference.

        At train time, samples *once per unique time step* so all documents at
        the same time step share the same α^(t) sample — matching the global-latent
        semantics of α (one set of topic embeddings per year, not per document).
        Per-document independent noise would add destructive gradient variance
        since α^(t) is identical for every document from year t.

        FIX (was): eps = torch.randn_like(std)  ← shape (B, K, L), different per doc
        FIX (now): eps drawn once per unique time step and broadcast to all docs at t
        """
        if not self.training:
            return self.alpha_mu[time_indices]  # (B, K, L) — posterior mean, no noise

        # Allocate output buffer (inherits shape, device, dtype from alpha_mu)
        alpha_samples = torch.empty(
            len(time_indices), self.alpha_mu.shape[1], self.alpha_mu.shape[2],
            device=self.alpha_mu.device, dtype=self.alpha_mu.dtype
        )

        # Sample ONCE per unique time step; broadcast to all docs at that time step
        for t in time_indices.unique():
            mask     = time_indices == t
            mu_t     = self.alpha_mu[t]               # (K, L)
            logvar_t = self.alpha_logvar[t]            # (K, L)
            std_t    = torch.exp(0.5 * logvar_t)       # (K, L)
            eps      = torch.randn_like(std_t)          # (K, L) — ONE sample for all docs at t
            alpha_samples[mask] = mu_t + eps * std_t    # broadcast to all docs at time t

        return alpha_samples  # (B, K, L)
    
    def compute_alpha_kl(self) -> torch.Tensor:
        """
        KL divergence for topic embedding mean-field parameters.
        
        p(α_0) = N(0, γ²I),  p(α_t | α_{t-1}) = N(α_{t-1}, γ²I)
        
        KL(N(μ, σ²) || N(m, γ²)) = 0.5 * [σ²/γ² + (μ-m)²/γ² - 1 - log(σ²/γ²)]
        
        For t > 0, α_{t-1} is itself random under q = N(μ_{t-1}, σ²_{t-1}), so we
        take the expectation E_{q(α_{t-1})}[KL(q(α_t) || N(α_{t-1}, γ²I))].
        The quadratic term expands as:
            E[(μ_t - α_{t-1})²] = (μ_t - μ_{t-1})² + σ²_{t-1}
        The σ²_{t-1}/γ² term was missing in the previous implementation, causing
        KL_α to be underestimated by ~2–3x and α to be under-regularized.
        
        Returns a raw sum over all T×K×L elements.
        Caller divides by num_train_docs to normalize to per-document scale.
        """
        gamma_sq = self.config.ALPHA_PRIOR_VARIANCE  # γ²
        total_kl = 0.0
        
        # t = 0: KL(q(α_0) || N(0, γ²I))
        mu_0  = self.alpha_mu[0]
        var_0 = self.alpha_logvar[0].exp()
        total_kl += 0.5 * torch.sum(
            var_0 / gamma_sq + mu_0.pow(2) / gamma_sq - 1.0 - torch.log(var_0 / gamma_sq)
        )
        
        # t > 0: E_{q(α_{t-1})}[KL(q(α_t) || N(α_{t-1}, γ²I))]
        # FIX: include σ²_{t-1}/γ² — the variance of the random prior mean α_{t-1}.
        for t in range(1, self.num_time_steps):
            mu_t     = self.alpha_mu[t]
            var_t    = self.alpha_logvar[t].exp()
            mu_prev  = self.alpha_mu[t - 1]
            var_prev = self.alpha_logvar[t - 1].exp()  # FIX: was missing entirely
            total_kl += 0.5 * torch.sum(
                var_t / gamma_sq
                + ((mu_t - mu_prev).pow(2) + var_prev) / gamma_sq  # FIX: + var_prev
                - 1.0
                - torch.log(var_t / gamma_sq)
            )
        
        return total_kl
    
    def forward(
        self,
        bow: torch.Tensor,
        time_indices: torch.Tensor,
        compute_loss: bool = True
    ) -> Dict[str, torch.Tensor]:
        """
        Forward pass through DETM.
        
        The LSTM is re-run on every forward call so that the full computation
        graph is fresh each batch — this matches the original Dieng et al.
        implementation and avoids the 'backward through graph a second time' error.
        
        Args:
            bow:          Bag-of-words (batch_size, V)
            time_indices: Time step index per document (batch_size,)
            compute_loss: Whether to compute ELBO loss terms
        
        Returns:
            Dict with theta, word_dist, and optionally all loss components.
            kl_eta and kl_alpha in the output dict are already normalized by
            num_train_docs (i.e. per-document scale), matching recon_loss and kl_theta.
        """
        # Run LSTM over all time steps — fresh graph every call, all gradients intact
        eta_timeline, kl_eta = self.temporal_baseline_encoder(self.avg_bow_timeline)
        
        # Look up η_{t_d} for each document (live tensor, gradient flows to LSTM)
        eta_batch = eta_timeline[time_indices]  # (B, K)
        
        # Encode: (BoW, η) → θ (logistic-normal)
        theta, theta_mu, theta_logvar = self.doc_encoder(bow, eta_batch)  # (B, K)
        
        # Sample α^(t) for each document's time step
        alpha_batch = self.sample_alpha(time_indices)  # (B, K, L)
        
        # Decode: (θ, α) → word distribution
        word_dist = self.decoder(theta, alpha_batch)  # (B, V)
        
        output = {
            'theta': theta,
            'theta_mu': theta_mu,
            'theta_logvar': theta_logvar,
            'word_dist': word_dist,
            'alpha': alpha_batch
        }
        
        if compute_loss:
            # 1. Reconstruction loss: -E_q[log p(w | θ, α, ρ)]  — per-document mean
            recon_loss = self.recon_loss_fn(bow, word_dist)
            
            # 2. KL(q(θ) || p(θ=η_t, I))
            #    Prior is Logistic-Normal(η_{t_d}, I), so the prior mean is η_batch,
            #    NOT zero. Without this, η has no role in the generative model.
            #    KL(N(μ, σ²) || N(η, I)) = 0.5 * [(μ-η)² + σ² - logσ² - 1]
            var_theta = theta_logvar.exp()
            kl_theta = 0.5 * torch.sum(
                (theta_mu - eta_batch).pow(2) + var_theta - theta_logvar - 1.0,
                dim=-1
            ).mean()
            
            # 3. KL(q(η) || p(η)) — normalize raw sum to per-document scale.
            kl_eta_normalized = kl_eta / self.num_train_docs

            # 4. KL(q(α) || p(α)) — normalize raw sum to per-document scale.
            kl_alpha = self.compute_alpha_kl()
            kl_alpha_normalized = kl_alpha / self.num_train_docs
            
            loss = (
                self.config.RECON_WEIGHT    * recon_loss +
                self.config.KL_THETA_WEIGHT * kl_theta +
                self.config.KL_ETA_WEIGHT   * kl_eta_normalized +
                self.config.KL_ALPHA_WEIGHT * kl_alpha_normalized
            )
            
            # Store normalized values so all logged/printed metrics are on the
            # same per-document scale and sum to the total loss.
            output.update({
                'loss': loss,
                'recon_loss': recon_loss,
                'kl_theta': kl_theta,
                'kl_eta': kl_eta_normalized,
                'kl_alpha': kl_alpha_normalized
            })
        
        return output
    
    def get_topics(self, time_idx: int = -1, top_n: int = 10) -> List[List[Tuple[str, float]]]:
        """Get top-N words per topic at a given time step."""
        if not hasattr(self, 'idx2word'):
            raise ValueError("idx2word not set. Assign model.idx2word before calling get_topics()")
        
        with torch.no_grad():
            if time_idx == -1:
                time_idx = self.num_time_steps - 1
            alpha = self.alpha_mu[time_idx]      # (K, L) — use mean for inspection
            beta  = self.decoder.get_beta(alpha) # (K, V)
            beta  = beta.cpu().numpy()
        
        topics = []
        for k in range(self.config.NUM_TOPICS):
            top_idx   = beta[k].argsort()[-top_n:][::-1]
            top_words = [(self.idx2word[idx], float(beta[k, idx])) for idx in top_idx]
            topics.append(top_words)
        return topics
    
    def get_document_topics(self, bow: torch.Tensor, time_indices: torch.Tensor) -> np.ndarray:
        """Get topic distributions for a batch of documents."""
        self.eval()
        device = next(self.parameters()).device
        bow = bow.to(device)
        time_indices = time_indices.to(device)
        with torch.no_grad():
            eta_timeline, _ = self.temporal_baseline_encoder(self.avg_bow_timeline)
            eta_batch = eta_timeline[time_indices]
            theta, _, _ = self.doc_encoder(bow, eta_batch)
        return theta.cpu().numpy()

print("DETM class defined.")


DETM class defined.


---
## 8. Dataset and DataLoader

In [17]:
class DETMDataset(Dataset):
    """PyTorch Dataset for DETM with temporal information."""
    
    def __init__(
        self,
        bow_matrix: np.ndarray,
        time_indices: np.ndarray,
        metadata: pd.DataFrame = None
    ):
        self.bow_matrix = torch.FloatTensor(bow_matrix)
        self.time_indices = torch.LongTensor(time_indices)
        self.metadata = metadata
    
    def __len__(self) -> int:
        return len(self.bow_matrix)
    
    def __getitem__(self, idx: int) -> Dict:
        item = {
            'bow': self.bow_matrix[idx],
            'time_idx': self.time_indices[idx]
        }
        
        if self.metadata is not None:
            item['metadata'] = self.metadata.iloc[idx].to_dict()
        
        return item

def create_dataloaders(
    processed_data: Dict,
    batch_size: int,
    train_split: float = 0.8,
    val_split: float = 0.1
) -> Tuple[DataLoader, DataLoader, DataLoader]:
    """
    Create train/val/test dataloaders with temporal information.
    
    Args:
        processed_data: Output from DataPreprocessor (includes temporal_info)
        batch_size: Batch size
        train_split: Fraction for training
        val_split: Fraction for validation
    
    Returns:
        train_loader, val_loader, test_loader
    """
    bow_matrix = processed_data['bow_matrix']
    metadata = processed_data['metadata']
    time_indices = processed_data['temporal_info']['doc_to_time']
    
    # Split indices
    n = len(bow_matrix)
    n_train = int(n * train_split)
    n_val = int(n * val_split)
    
    # Create datasets
    train_dataset = DETMDataset(
        bow_matrix[:n_train],
        time_indices[:n_train],
        metadata.iloc[:n_train] if metadata is not None else None
    )
    
    val_dataset = DETMDataset(
        bow_matrix[n_train:n_train+n_val],
        time_indices[n_train:n_train+n_val],
        metadata.iloc[n_train:n_train+n_val] if metadata is not None else None
    )
    
    test_dataset = DETMDataset(
        bow_matrix[n_train+n_val:],
        time_indices[n_train+n_val:],
        metadata.iloc[n_train+n_val:] if metadata is not None else None
    )
    
    # Create dataloaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0
    )
    
    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0
    )
    
    print(f"\nDataset splits:")
    print(f"  Train: {len(train_dataset)} documents")
    print(f"  Val: {len(val_dataset)} documents")
    print(f"  Test: {len(test_dataset)} documents")
    
    return train_loader, val_loader, test_loader

print("Dataset classes defined.")

Dataset classes defined.


---
## 9. Training Infrastructure

### 9.1 DETM Trainer Class

In [18]:
class DETMTrainer:
    """
    Training loop for DETM with monitoring and checkpointing.
    Includes TensorBoard integration for real-time visualization.
    
    Optimizer: Adam with constant lr=0.001 and wd=1.2e-6 (matches Dieng et al. 2019).
    No LR scheduler — constant learning rate avoids premature convergence.
    Each training run gets a timestamped TensorBoard subdirectory so runs can
    be compared side-by-side in TensorBoard.
    """
    
    def __init__(
        self,
        model: DETM,
        config: Config,
        train_loader: DataLoader,
        val_loader: DataLoader,
        device: torch.device,
        log_dir: Optional[str] = None
    ):
        from datetime import datetime
        
        self.model = model.to(device)
        self.config = config
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.device = device
        
        # Each run gets its own timestamped subdirectory so TensorBoard can
        # overlay multiple runs for comparison (compare runs sidebar).
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        if log_dir is None:
            log_dir = str(config.OUTPUTS_DIR / 'tensorboard_logs' / timestamp)
        self.writer = SummaryWriter(log_dir=log_dir)
        print(f"TensorBoard run: {timestamp}")
        print(f"  Logging to: {log_dir}")
        print(f"  View with: tensorboard --logdir={config.OUTPUTS_DIR / 'tensorboard_logs'}")
        
        # Adam with constant lr and small weight decay — from Dieng et al. 2019.
        # AdamW is NOT used because AdamW's decoupled weight decay differs from
        # the L2 regularization the paper uses with plain Adam.
        self.optimizer = Adam(
            model.parameters(),
            lr=config.LEARNING_RATE,
            weight_decay=config.WEIGHT_DECAY
        )
        # No LR scheduler — constant lr=0.001 throughout (matches paper).
        
        # Tracking
        self.global_step = 0
        self.history = {
            'train_loss': [],
            'train_recon': [],
            'train_kl_theta': [],
            'train_kl_eta': [],
            'train_kl_alpha': [],
            'val_loss': [],
            'val_recon': [],
            'val_kl_theta': [],
            'val_kl_eta': [],
            'val_kl_alpha': [],
            'learning_rate': []
        }
        
        self.best_val_loss = float('inf')
        self.patience_counter = 0
    
    def train_epoch(self) -> Dict[str, float]:
        """Train for one epoch."""
        self.model.train()
        
        total_loss = 0
        total_recon = 0
        total_kl_theta = 0
        total_kl_eta = 0
        total_kl_alpha = 0
        num_batches = 0
        
        pbar = tqdm(self.train_loader, desc="Training")
        for batch in pbar:
            bow = batch['bow'].to(self.device)
            time_indices = batch['time_idx'].to(self.device)
            
            # Forward pass — LSTM re-run each batch for correct gradient flow
            output = self.model(bow, time_indices, compute_loss=True)
            
            loss = output['loss']
            recon_loss = output['recon_loss']
            kl_theta = output['kl_theta']
            kl_eta = output['kl_eta']
            kl_alpha = output['kl_alpha']
            
            # Backward pass
            self.optimizer.zero_grad()
            loss.backward()
            
            # Gradient clipping (paper uses 2.0)
            torch.nn.utils.clip_grad_norm_(
                self.model.parameters(),
                self.config.CLIP_GRAD
            )
            
            # Optimizer step
            self.optimizer.step()
            
            # Accumulate metrics
            total_loss += loss.item()
            total_recon += recon_loss.item()
            total_kl_theta += kl_theta.item()
            total_kl_eta += kl_eta.item()
            total_kl_alpha += kl_alpha.item()
            num_batches += 1
            
            # Log batch metrics to TensorBoard (every 10 batches)
            if self.global_step % 10 == 0:
                self.writer.add_scalar('Batch/Loss', loss.item(), self.global_step)
                self.writer.add_scalar('Batch/Reconstruction', recon_loss.item(), self.global_step)
                self.writer.add_scalar('Batch/KL_theta', kl_theta.item(), self.global_step)
                self.writer.add_scalar('Batch/KL_eta', kl_eta.item(), self.global_step)
                self.writer.add_scalar('Batch/KL_alpha', kl_alpha.item(), self.global_step)
            
            self.global_step += 1
            
            # Update progress bar
            pbar.set_postfix({
                'loss': f"{loss.item():.2f}",
                'recon': f"{recon_loss.item():.2f}",
                'KL_θ': f"{kl_theta.item():.2f}",
                'KL_η': f"{kl_eta.item():.2f}",
                'KL_α': f"{kl_alpha.item():.2f}"
            })
        
        return {
            'train_loss': total_loss / num_batches,
            'train_recon': total_recon / num_batches,
            'train_kl_theta': total_kl_theta / num_batches,
            'train_kl_eta': total_kl_eta / num_batches,
            'train_kl_alpha': total_kl_alpha / num_batches
        }
    
    @torch.no_grad()
    def validate(self) -> Dict[str, float]:
        """Validate on validation set."""
        self.model.eval()
        
        total_loss = 0
        total_recon = 0
        total_kl_theta = 0
        total_kl_eta = 0
        total_kl_alpha = 0
        num_batches = 0
        
        for batch in tqdm(self.val_loader, desc="Validation"):
            bow = batch['bow'].to(self.device)
            time_indices = batch['time_idx'].to(self.device)
            
            output = self.model(bow, time_indices, compute_loss=True)
            
            total_loss += output['loss'].item()
            total_recon += output['recon_loss'].item()
            total_kl_theta += output['kl_theta'].item()
            total_kl_eta += output['kl_eta'].item()
            total_kl_alpha += output['kl_alpha'].item()
            num_batches += 1
        
        return {
            'val_loss': total_loss / num_batches,
            'val_recon': total_recon / num_batches,
            'val_kl_theta': total_kl_theta / num_batches,
            'val_kl_eta': total_kl_eta / num_batches,
            'val_kl_alpha': total_kl_alpha / num_batches
        }
    
    def save_checkpoint(self, epoch: int, is_best: bool = False):
        """Save model checkpoint."""
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': self.model.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'best_val_loss': self.best_val_loss,
            'history': self.history,
            'config': self.config
        }
        
        # Save regular checkpoint
        if epoch % self.config.SAVE_EVERY == 0:
            path = self.config.MODELS_DIR / f'detm_epoch_{epoch}.pt'
            torch.save(checkpoint, path)
            print(f"Checkpoint saved: {path}")
        
        # Save best model
        if is_best:
            path = self.config.MODELS_DIR / 'detm_best.pt'
            torch.save(checkpoint, path)
            print(f"Best model saved: {path}")
    
    def train(self, num_epochs: int = None):
        """Complete training loop."""
        num_epochs = num_epochs or self.config.NUM_EPOCHS
        current_lr = self.config.LEARNING_RATE  # constant throughout
        
        print("\n" + "=" * 60)
        print("TRAINING DETM")
        print("=" * 60)
        print(f"Number of epochs: {num_epochs}")
        print(f"Learning rate: {current_lr} (constant — no scheduler)")
        print(f"Optimizer: Adam  |  weight_decay: {self.config.WEIGHT_DECAY}")
        print(f"Grad clip: {self.config.CLIP_GRAD}")
        print(f"Device: {self.device}")
        print("=" * 60 + "\n")
        
        for epoch in range(1, num_epochs + 1):
            print(f"\nEpoch {epoch}/{num_epochs}")
            print("-" * 60)
            
            # Train
            train_metrics = self.train_epoch()
            
            # Validate
            val_metrics = self.validate()
            
            # Track metrics in history
            self.history['train_loss'].append(train_metrics['train_loss'])
            self.history['train_recon'].append(train_metrics['train_recon'])
            self.history['train_kl_theta'].append(train_metrics['train_kl_theta'])
            self.history['train_kl_eta'].append(train_metrics['train_kl_eta'])
            self.history['train_kl_alpha'].append(train_metrics['train_kl_alpha'])
            self.history['val_loss'].append(val_metrics['val_loss'])
            self.history['val_recon'].append(val_metrics['val_recon'])
            self.history['val_kl_theta'].append(val_metrics['val_kl_theta'])
            self.history['val_kl_eta'].append(val_metrics['val_kl_eta'])
            self.history['val_kl_alpha'].append(val_metrics['val_kl_alpha'])
            self.history['learning_rate'].append(current_lr)
            
            # Log epoch metrics to TensorBoard
            self.writer.add_scalar('Epoch/Train/Loss', train_metrics['train_loss'], epoch)
            self.writer.add_scalar('Epoch/Train/Reconstruction', train_metrics['train_recon'], epoch)
            self.writer.add_scalar('Epoch/Train/KL_theta', train_metrics['train_kl_theta'], epoch)
            self.writer.add_scalar('Epoch/Train/KL_eta', train_metrics['train_kl_eta'], epoch)
            self.writer.add_scalar('Epoch/Train/KL_alpha', train_metrics['train_kl_alpha'], epoch)
            
            self.writer.add_scalar('Epoch/Val/Loss', val_metrics['val_loss'], epoch)
            self.writer.add_scalar('Epoch/Val/Reconstruction', val_metrics['val_recon'], epoch)
            self.writer.add_scalar('Epoch/Val/KL_theta', val_metrics['val_kl_theta'], epoch)
            self.writer.add_scalar('Epoch/Val/KL_eta', val_metrics['val_kl_eta'], epoch)
            self.writer.add_scalar('Epoch/Val/KL_alpha', val_metrics['val_kl_alpha'], epoch)
            
            self.writer.add_scalar('Epoch/Learning_Rate', current_lr, epoch)
            
            # Log gradient norms
            total_norm = 0.0
            for p in self.model.parameters():
                if p.grad is not None:
                    param_norm = p.grad.data.norm(2)
                    total_norm += param_norm.item() ** 2
            total_norm = total_norm ** 0.5
            self.writer.add_scalar('Epoch/Gradient_Norm', total_norm, epoch)
            
            # Print summary
            print(f"\nEpoch {epoch} Summary:")
            print(f"  Train Loss: {train_metrics['train_loss']:.4f} "
                  f"(Recon: {train_metrics['train_recon']:.4f}, "
                  f"KL_θ: {train_metrics['train_kl_theta']:.4f}, "
                  f"KL_η: {train_metrics['train_kl_eta']:.4f}, "
                  f"KL_α: {train_metrics['train_kl_alpha']:.4f})")
            print(f"  Val Loss: {val_metrics['val_loss']:.4f} "
                  f"(Recon: {val_metrics['val_recon']:.4f}, "
                  f"KL_θ: {val_metrics['val_kl_theta']:.4f}, "
                  f"KL_η: {val_metrics['val_kl_eta']:.4f}, "
                  f"KL_α: {val_metrics['val_kl_alpha']:.4f})")
            print(f"  Learning Rate: {current_lr:.6f}")
            
            # Check for improvement
            is_best = val_metrics['val_loss'] < self.best_val_loss
            if is_best:
                self.best_val_loss = val_metrics['val_loss']
                self.patience_counter = 0
                print(f"  ✓ New best validation loss: {self.best_val_loss:.4f}")
            else:
                self.patience_counter += 1
                print(f"  No improvement for {self.patience_counter} epochs")
            
            # Save checkpoint
            self.save_checkpoint(epoch, is_best=is_best)
            
            # Early stopping
            if self.patience_counter >= self.config.PATIENCE:
                print(f"\nEarly stopping triggered after {epoch} epochs")
                break
        
        # Close TensorBoard writer
        self.writer.close()
        print(f"\nTensorBoard logs saved. View with: tensorboard --logdir={self.writer.log_dir}")
        
        print("\n" + "=" * 60)
        print("TRAINING COMPLETE")
        print("=" * 60)
        print(f"Best validation loss: {self.best_val_loss:.4f}")
        print("=" * 60)
        
        return self.history


---
## 10. Evaluation Metrics

### 10.1 Topic Coherence and Quality Metrics

In [19]:
class TopicEvaluator:
    """
    Evaluates topic model quality using various metrics.
    
    Metrics:
    - Topic coherence (C_v, C_npmi)
    - Topic diversity
    - Perplexity
    """
    
    def __init__(self, tokens_list: List[List[str]], vocabulary: List[str]):
        self.tokens_list = tokens_list
        self.vocabulary = vocabulary
        
        # Create Gensim dictionary and corpus
        self.dictionary = Dictionary(tokens_list)
        self.corpus = [self.dictionary.doc2bow(doc) for doc in tokens_list]
    
    def compute_coherence(
        self,
        topics: List[List[str]],
        coherence_type: str = 'c_v'
    ) -> float:
        """
        Compute topic coherence score.
        
        Args:
            topics: List of topics, each is a list of words
            coherence_type: 'c_v' or 'c_npmi'
        
        Returns:
            Average coherence score
        """
        cm = CoherenceModel(
            topics=topics,
            texts=self.tokens_list,
            dictionary=self.dictionary,
            coherence=coherence_type
        )
        
        return cm.get_coherence()
    
    def compute_topic_diversity(self, topics: List[List[str]]) -> float:
        """
        Compute topic diversity.
        
        Measures what fraction of unique words appear in the top-N words
        across all topics. Higher is better (less redundancy).
        
        Args:
            topics: List of topics, each is a list of words
        
        Returns:
            Diversity score (0 to 1)
        """
        unique_words = set()
        total_words = 0
        
        for topic in topics:
            unique_words.update(topic)
            total_words += len(topic)
        
        diversity = len(unique_words) / total_words if total_words > 0 else 0
        return diversity
    
    def evaluate_topics(
        self,
        model: DETM,
        top_n_words: List[int] = [10, 15, 20]
    ) -> Dict[str, float]:
        """
        Comprehensive topic evaluation.
        
        Args:
            model: Trained DETM model
            top_n_words: List of N values for top-N words
        
        Returns:
            Dictionary of evaluation metrics
        """
        results = {}
        
        for n in top_n_words:
            # Get topics
            topics_with_probs = model.get_topics(top_n=n)
            topics = [[word for word, _ in topic] for topic in topics_with_probs]
            
            # Compute metrics
            cv_score = self.compute_coherence(topics, coherence_type='c_v')
            npmi_score = self.compute_coherence(topics, coherence_type='c_npmi')
            diversity = self.compute_topic_diversity(topics)
            
            results[f'coherence_cv_top{n}'] = cv_score
            results[f'coherence_npmi_top{n}'] = npmi_score
            results[f'diversity_top{n}'] = diversity
        
        return results
    
    @staticmethod
    def compute_perplexity(
        model: DETM,
        data_loader: DataLoader,
        device: torch.device
    ) -> float:
        """
        Compute perplexity on a dataset.
        
        Perplexity = exp(-log likelihood / num_words)
        
        Args:
            model: Trained DETM model
            data_loader: DataLoader for dataset
            device: Device to run on
        
        Returns:
            Perplexity score
        """
        model.eval()
        
        total_nll = 0
        total_words = 0
        
        with torch.no_grad():
            for batch in data_loader:
                bow = batch['bow'].to(device)
                time_indices = batch['time_idx'].to(device)
                
                output = model(bow, time_indices, compute_loss=True)
                
                # Accumulated negative log-likelihood
                total_nll += output['recon_loss'].item() * len(bow)
                total_words += bow.sum().item()
        
        # Perplexity = exp(average_nll)
        perplexity = np.exp(total_nll / total_words)
        
        return perplexity

print("TopicEvaluator class defined.")

TopicEvaluator class defined.


---
## 12. Main Execution Pipeline

### Run complete DETM workflow

In [20]:
# ── Pipeline control ──────────────────────────────────────────────────────────
# Set LOAD_PRETRAINED = True to skip training and load the best saved checkpoint.
# NOTE: Must retrain from scratch after bug fixes — the old checkpoint has
#       collapsed alpha parameters that cannot be recovered by fine-tuning.
LOAD_PRETRAINED = True
CHECKPOINT_PATH = config.MODELS_DIR / 'detm_best.pt'
# ─────────────────────────────────────────────────────────────────────────────

# Step 1: Load and preprocess data
df_raw = load_and_explore_data()
preprocessor = DataPreprocessor(config)
processed_data = preprocessor.preprocess_corpus(df_raw)

# Step 2: Generate embeddings
# Word2Vec is trained on first run and cached to data/word2vec.model.
# Subsequent runs load from cache — no retraining.
# Pass tokens_list so training can proceed if no cache exists yet.
embedding_gen = EmbeddingGenerator(config)
word_embeddings = embedding_gen.generate_vocabulary_embeddings(
    preprocessor.vocabulary,
    tokens_list=processed_data['tokens_list'],  # ignored when cached model exists
)
# Keep on CPU — word_embeddings is stored as nn.Parameter inside DETMDecoder,
# and will be moved to device automatically when model.to(device) is called.
word_embeddings_tensor = torch.FloatTensor(word_embeddings)

# Step 3: Create dataloaders (must happen before model init to get num_train_docs)
train_loader, val_loader, test_loader = create_dataloaders(
    processed_data,
    batch_size=config.BATCH_SIZE
)

# Step 4: Initialize model (architecture must always be constructed first)
model = DETM(
    config,
    word_embeddings_tensor,
    processed_data['temporal_info']['num_time_steps'],
    processed_data['temporal_info']['avg_bow_per_time'],
    num_train_docs=len(train_loader.dataset)  # normalizes kl_eta and kl_alpha per document
)
model.idx2word = preprocessor.idx2word
print(f"\nModel initialized with {sum(p.numel() for p in model.parameters())} parameters")
print(f"  num_train_docs = {model.num_train_docs}")

# Step 5: Train or load checkpoint
if LOAD_PRETRAINED:
    if not CHECKPOINT_PATH.exists():
        raise FileNotFoundError(
            f"No checkpoint found at {CHECKPOINT_PATH}. "
            "Train the model first (set LOAD_PRETRAINED = False)."
        )
    # weights_only=False is required because the checkpoint contains a Config object.
    # This is safe since the checkpoint was produced by this notebook.
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    model = model.to(device)
    history = checkpoint.get('history', {})
    print(f"\nLoaded checkpoint: {CHECKPOINT_PATH}")
    print(f"  Saved at epoch  : {checkpoint.get('epoch', 'unknown')}")
    print(f"  Best val loss   : {checkpoint.get('best_val_loss', float('nan')):.4f}")
else:
    model = model.to(device)
    trainer = DETMTrainer(model, config, train_loader, val_loader, device)
    history = trainer.train()

    # ── FIX: reload the best checkpoint for evaluation ────────────────────────
    # trainer.train() mutates `model` in-place; at completion it holds the LAST
    # epoch weights (e.g. epoch 83), NOT the best-val-loss epoch (~epoch 68).
    # The best model was saved to disk during training — reload it now so that
    # all downstream evaluation uses the genuinely best weights.
    if CHECKPOINT_PATH.exists():
        best_ckpt = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
        model.load_state_dict(best_ckpt['model_state_dict'])
        print(f"\n✓ Reloaded best checkpoint for evaluation"
              f" (epoch {best_ckpt.get('epoch', '?')},"
              f" val_loss={best_ckpt.get('best_val_loss', float('nan')):.4f})")
    else:
        print("\n⚠ No best checkpoint found on disk — evaluating last-epoch weights.")
    # ─────────────────────────────────────────────────────────────────────────

# Step 6: Evaluate
evaluator = TopicEvaluator(processed_data['tokens_list'], preprocessor.vocabulary)
metrics = evaluator.evaluate_topics(model, top_n_words=config.TOP_N_WORDS)
perplexity = evaluator.compute_perplexity(model, test_loader, device)

print("\nEvaluation Metrics:")
for metric, value in metrics.items():
    print(f"  {metric}: {value:.4f}")
print(f"  Test Perplexity: {perplexity:.4f}")

# Step 7: Save results
results = {
    'config': config.__dict__,
    'metrics': metrics,
    'perplexity': perplexity,
    'history': history
}
with open(config.OUTPUTS_DIR / 'results.json', 'w') as f:
    json.dump(results, f, indent=2, default=str)

print("\nPipeline complete!")


UN GENERAL DEBATES DATASET

Dataset shape: (7507, 4)

Columns: ['session', 'year', 'country', 'text']

Data types:
session     int64
year        int64
country    object
text       object
dtype: object

Missing values:
session    0
year       0
country    0
text       0
dtype: int64

Year range: 1970 - 2015
Years in dataset: [np.int64(1970), np.int64(1971), np.int64(1972), np.int64(1973), np.int64(1974), np.int64(1975), np.int64(1976), np.int64(1977), np.int64(1978), np.int64(1979), np.int64(1980), np.int64(1981), np.int64(1982), np.int64(1983), np.int64(1984), np.int64(1985), np.int64(1986), np.int64(1987), np.int64(1988), np.int64(1989), np.int64(1990), np.int64(1991), np.int64(1992), np.int64(1993), np.int64(1994), np.int64(1995), np.int64(1996), np.int64(1997), np.int64(1998), np.int64(1999), np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), n

Tokenizing:   0%|          | 0/7507 [00:00<?, ?it/s]

Documents after cleaned-token length filter (≥15): 7507 (removed 0)
Building vocabulary...


Counting frequencies:   0%|          | 0/7507 [00:00<?, ?it/s]


Vocabulary size: 10000
Total unique tokens before filtering: 42959
Tokens passing df gates [30, 2252]: 10108
Final vocabulary (top-10000 by IDF): 10000

Converting to bag-of-words representation...


Creating BoW:   0%|          | 0/7507 [00:00<?, ?it/s]


Final number of documents: 7507
Vocabulary size: 10000
Average vocab words per document: 496.74
Sparsity: 96.19%
Documents sorted by year: 1970 → 2015

TEMPORAL STRUCTURE
Number of time steps (T): 46
Time range: 1970 - 2015
Average documents per time step: 163.2
Min docs in time step: 70
Max docs in time step: 195

Preprocessed data saved to: ../data/preprocessed_data.pkl
✓ Cached Word2Vec model found — loading (skip training).
  To retrain, call with force_retrain=True or delete ../data/word2vec.model
✓ Word2Vec loaded from cache: ../data/word2vec.model (11,084 words)

EXTRACTING VOCABULARY EMBEDDINGS
Vocabulary size: 10,000
✓ All vocabulary words found in Word2Vec model (0 OOV)

Embedding matrix shape: (10000, 300)
Embedding statistics (before L2-norm):
  Mean: -0.0061
  Std:  0.2275
  Min:  -1.4018
  Max:  1.3503
✓ Embeddings saved to: ../data/word_embeddings.npy

Dataset splits:
  Train: 6005 documents
  Val: 750 documents
  Test: 752 documents

Model initialized with 22320000 par

---
## 14. Next Steps and Extensions

### Possible extensions:
1. **Temporal Analysis**: Track how specific topics evolve year-by-year
2. **Country-level Analysis**: Compare topic distributions across countries
3. **Dynamic Coherence**: Measure coherence changes over time
4. **Transfer to Financial Data**: Adapt this implementation for financial text
5. **Hyperparameter Optimization**: Grid search or Bayesian optimization
6. **Comparison with Baselines**: LDA, Static ETM, etc.

### Paper Reproducibility Checklist:
- [ ] Match model architecture (encoder layers, dimensions)
- [ ] Use same hyperparameters (learning rate, batch size, epochs)
- [ ] Verify embedding quality (Word2Vec vs BERT comparison)
- [ ] Compare coherence scores with paper benchmarks
- [ ] Validate perplexity on similar datasets
- [ ] Qualitative topic inspection

### Notes:
- **Modular Design**: Each component (encoder, decoder, losses) is self-contained
- **Algorithm Boundaries**: Clear separation between generative model, inference network, and temporal dynamics
- **Reproducibility**: All random seeds set, configurations saved
- **Extensibility**: Easy to swap components (e.g., different transition models)